In [0]:
from confluent_kafka import Producer
import json
import time

class InvoiceProducer:
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath(".")
        self.topic = "invoices"
        self.conf = {'bootstrap.servers': dbutils.secrets.get(scope="kafka-secrets",key="bootstrap-server"),
                     'security.protocol': 'SASL_SSL',
                     'sasl.mechanism': 'PLAIN',
                     'sasl.username': dbutils.secrets.get(scope="kafka-secrets",key="cluster-api-key"),
                     'sasl.password': dbutils.secrets.get(scope="kafka-secrets",key="cluster-api-secret"),
                     'client.id': "nirajrkhapne@gmail.com"}
        
    def delivery_callback(self, err, msg):
        if err:
            print('ERROR: Message failed delivery: {}'.format(err))
        else:
            key = msg.key().decode('utf-8')
            invoice_id = json.loads(msg.value().decode('utf-8'))["InvoiceNumber"]
            print(f"Produced event to : key = {key} value = {invoice_id}")
        
    def produce_invoices(self, producer,counts):
        counter = 0
        with open(f"{self.base_data_dir}/data/invoices.json") as lines:
            for line in lines:
                invoice = json.loads(line)
                store_id = invoice["StoreID"]
                producer.produce(self.topic
                                 , key=store_id
                                 , value=json.dumps(invoice)
                                 , callback=self.delivery_callback)
                time.sleep(0.5)
                producer.poll(1)
                counter+=1
                if counter == counts:
                    break

    def start(self):
        kafka_producer = Producer(self.conf)
        self.produce_invoices(kafka_producer,20)
        kafka_producer.flush(20)


if __name__ == "__main__":
    invoice_producer = InvoiceProducer()
    invoice_producer.start()